In [2]:
from tqdm import tqdm
import jiwer
import os
import torch
from faster_whisper import WhisperModel, BatchedInferencePipeline

/home/ming/anaconda3/envs/whisper/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
txt_dir = "../data/converted/thai-central-to-vctk/txt"
wav_dir = "../data/converted/thai-central-to-vctk/wav16_silence_trimmed"

In [4]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "large-v3"

model = WhisperModel(model_id, device="cuda")
batched_model = BatchedInferencePipeline(model=model)

In [5]:
class CustomDataset:
    def __init__(self, txt_dir, wav_dir):
        self.txt_dir = txt_dir
        self.wav_dir = wav_dir
        self.txts = []
        self.wav_files = []
        for speaker in tqdm(os.listdir(txt_dir)):
            for filename in os.listdir(os.path.join(txt_dir, speaker)):
                filename = filename.split(".")[0]
                txt_file = os.path.join(txt_dir, speaker, filename + ".txt")
                wav_file = os.path.join(wav_dir, speaker, filename + ".flac")
                if os.path.exists(txt_file) and os.path.exists(wav_file):
                    with open(txt_file, "r") as f:
                        txt = f.read()
                    self.txts.append(txt)
                    self.wav_files.append(wav_file)

    def __len__(self):
        return len(self.txts)

    def __getitem__(self, idx):
        return self.txts[idx], self.wav_files[idx]

In [7]:
dataset = CustomDataset(txt_dir, wav_dir)

100%|██████████| 984/984 [00:38<00:00, 25.55it/s]


In [8]:
print(dataset[0])

('ปู่ขายน้ำมัลเบอร์รี่หนึ่งร้อยเปอร์เซ็นวังภูหมอกฟาร์มขวดละหนึ่งแสนหนึ่งหมื่นบาทค่ะ', '../data/converted/thai-central-to-vctk/wav16_silence_trimmed/tc0864/thai-central_201353_mic1.flac')


In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

In [14]:
out_dir = './transcribed.txt'
batch_counter = 0
txts = []
transcripts = []
filenames = []

for batch in tqdm(dataset, desc="Transcribing"):
    try:
        transcribed, _ = batched_model.transcribe(batch[1], batch_size=64, beam_size=5)
        transcripts.extend(["".join(t.text.split()) for t in transcribed])
    except Exception as e:
        print(f"Error in batch {batch_counter}: {e}")
        transcripts.extend([-1])  # Fallback for err
    
    filenames.extend(batch[1])
    txts.extend(["".join(t.split()) for t in list(batch[0])])  # Assuming batch[0] contains reference texts

    # Write to file every 20 batches
    if batch_counter % 1000 == 0 and batch_counter > 0:
        with open(out_dir, "a") as f:
            for filename, txt, transcript in zip(filenames, txts, transcripts):
                f.write(f"{filename}|{txt}|{transcript}\n")
        
        txts.clear()
        transcripts.clear()
        filenames.clear()

    batch_counter += 1

# Write any remaining data after the loop
if txts:
    with open(out_dir, "a") as f:
        for filename, txt, transcript in zip(filenames, txts, transcripts):
            f.write(f"{filename}|{txt}|{transcript}\n")


Transcribing:   0%|          | 102/225028 [01:04<39:17:39,  1.59it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

df = pd.read_csv('transcribed.txt', sep='|', header=None)
df.columns = ['filename', 'txt', 'transcript']
df

In [ ]:
import jiwer
from tqdm import tqdm

cers = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    txt = row['txt']
    transcript = row['transcript']
    cer = jiwer.cer(txt, transcript)
    cers.append(cer)
    df.at[i, 'cer'] = cer

df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compute Q1, Q3, and IQR
Q1 = np.percentile(cers, 25)
Q3 = np.percentile(cers, 75)
IQR = Q3 - Q1

# Define outlier boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
cers_filtered = [x for x in cers if lower_bound <= x <= upper_bound]

# Plot histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={'width_ratios': [3, 1]})

# Left plot (75% width)
axes[0].hist(cers_filtered, bins=50, color='crimson', edgecolor='black')
axes[0].set_xlabel("Character Error Rate")
axes[0].set_ylabel("Number of Samples")
axes[0].set_title("Character Error Rate Distribution (Outliers Removed)")

# Right plot (25% width)
axes[1].bar(["Total", "Filtered"], [len(cers), len(cers_filtered)], color=['skyblue', 'crimson'], edgecolor='black')
axes[1].set_ylabel("Number of Samples")
axes[1].set_title("Total vs Filtered Samples")

plt.tight_layout()
plt.show()

print(f"Outliers removed: {len(cers) - len(cers_filtered)}")

In [ ]:
df.to_csv('transcribed.csv', index=False)

In [ ]:
speaker_n_files = {}
speaker_perfect_transcripts = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    speaker = row['filename'].split('/')[-2]
    if speaker not in speaker_n_files:
        speaker_n_files[speaker] = 1
        speaker_perfect_transcripts[speaker] = 0
    else:
        speaker_n_files[speaker] += 1
    
    if round(row['cer'], 2) == 0:
        speaker_perfect_transcripts[speaker] += 1

print(speaker_n_files)
print(speaker_perfect_transcripts)


In [ ]:
speaker_perfect_transcripts = dict(sorted(speaker_perfect_transcripts.items(), key=lambda x: x[1], reverse=True))
print(speaker_perfect_transcripts)

In [ ]:
for speaker in speaker_perfect_transcripts:
    if speaker_perfect_transcripts[speaker] >= 100:
        print(f"Speaker {speaker} has {speaker_perfect_transcripts[speaker]} perfect transcripts out of {speaker_n_files[speaker]} files")

In [ ]:
ratios = {}
for speaker in speaker_perfect_transcripts:
    ratios[speaker] = speaker_perfect_transcripts[speaker] / speaker_n_files[speaker]

# sort by value
ratios = dict(sorted(ratios.items(), key=lambda item: item[1], reverse=True))
print(ratios)

In [ ]:
# plt hist by speaker
mean_perfect_transcripts = np.mean(list(speaker_perfect_transcripts.values()))
plt.hist(speaker_perfect_transcripts.values(), bins=50)
plt.vlines(mean_perfect_transcripts, 0, 100, colors='r', linestyles='dashed', label=f'Mean: {mean_perfect_transcripts}')
plt.legend()
plt.xlabel("Number of Perfect Transcripts")
plt.ylabel("Frequency")
plt.title("Number of Perfect Transcripts by Speaker")
plt.show()

In [ ]:
qualified_speakers = []
for speaker in speaker_perfect_transcripts:
    n_perfect = speaker_perfect_transcripts[speaker]
    if n_perfect >= 20:
        qualified_speakers.append(speaker)

print(qualified_speakers)
print(len(qualified_speakers))

In [ ]:
qualified_files = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    speaker = row['filename'].split('/')[-2]
    if speaker in qualified_speakers and round(row['cer'], 2) == 0:
        qualified_files.append(row['filename'])

# save to file
with open('qualified_files.txt', 'w') as f:
    for file in qualified_files:
        f.write(f"{file}\n")


In [ ]:
print(len(qualified_files))